# E-Commerce Sales & Customer Intelligence — SQL Analysis

## Project Overview

This notebook performs SQL-based analysis on the Online Retail dataset to evaluate sales performance, product performance, customer behaviour, geographic contribution, customer segmentation, and retention.

The cleaned transactional dataset is loaded into a SQLite database and analysed using SQL queries. Python is used for database connectivity, query execution, validation, and exporting analytical results.

### Objectives

The analysis focuses on:

- Overall sales and revenue performance
- Monthly revenue trends
- Product performance
- Geographic sales contribution
- Customer behaviour and revenue concentration
- RFM customer segmentation
- Customer cohort retention
- Preparation of analytical datasets for Power BI

### Analytical Workflow

**Cleaned Data → SQLite Database → SQL Analysis → Analytical Outputs → Power BI Dashboard**

The SQL layer acts as the analytical bridge between the cleaned dataset and the final business intelligence dashboard.

In [2]:
import pandas as pd
import numpy as np
import sqlite3

In [4]:
df = pd.read_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\cleaned\online_retail_cleaned.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Year,Month,MonthName,YearMonth,TransactionType
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,December,2010-12,Sale
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,2010-12,Sale
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,December,2010-12,Sale
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,2010-12,Sale
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,2010-12,Sale


## 1. Database Setup

The cleaned Online Retail dataset is loaded into a SQLite database to enable structured SQL-based analysis.

The `online_retail` table contains transaction-level information including invoice details, products, quantities, prices, customers, countries, sales values, transaction types, and anomaly indicators.

The database is used as the primary analytical source for the SQL queries that follow.

In [7]:
conn = sqlite3.connect(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\ecommerce.db")

In [9]:
df.to_sql("online_retail",conn,if_exists="replace",index=False)

536641

In [11]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM online_retail",conn)

,total_rows
0,536641


## 2. Overall Sales Performance

This section establishes the core business KPIs using transaction-level sales data.

The analysis measures:

- Total revenue
- Total orders and invoices
- Average order value
- Sales versus non-sales transactions
- Overall transaction activity

These metrics provide the baseline for evaluating the overall performance of the e-commerce business.

In [14]:
query = """
SELECT
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM online_retail
WHERE TransactionType = 'Sale';
"""
pd.read_sql(query, conn)

,Total_Revenue
0,10619986.68


In [16]:
df[df["TransactionType"] == "Sale"]["Sales"].sum()

10619986.684

In [18]:
df["Sales"].sum()

9726006.953999998

In [20]:
df.groupby("TransactionType")["Sales"].sum()

TransactionType
Return   -8.939797e+05
Sale      1.061999e+07
Name: Sales, dtype: float64

In [22]:
query = """
SELECT
    COUNT(DISTINCT InvoiceNo) AS Total_Orders
FROM online_retail
WHERE TransactionType = 'Sale';
"""

pd.read_sql(query, conn)

,Total_Orders
0,20728


In [24]:
query = """
SELECT
    COUNT(DISTINCT InvoiceNo) AS Total_Invoices,
    COUNT(DISTINCT CASE
        WHEN TransactionType = 'Sale'
        THEN InvoiceNo
    END) AS Sales_Orders,
    COUNT(DISTINCT CASE
        WHEN TransactionType LIKE 'Return%'
        THEN InvoiceNo
    END) AS Return_Orders
FROM online_retail;
"""

pd.read_sql(query, conn)

,Total_Invoices,Sales_Orders,Return_Orders
0,25900,20728,5172


In [23]:
query = """
SELECT
    ROUND(
        SUM(Sales) /
        COUNT(DISTINCT InvoiceNo),
        2
    ) AS Average_Order_Value
FROM online_retail
WHERE TransactionType = 'Sale';
"""

pd.read_sql(query, conn)


,Average_Order_Value
0,512.35


In [25]:
query = """
SELECT
    ROUND(
        SUM(Sales) /
        COUNT(DISTINCT InvoiceNo),
        2
    ) AS Net_Average_Order_Value
FROM online_retail
WHERE TransactionType IN ('Sale', 'Return');
"""

pd.read_sql(query, conn)


,Net_Average_Order_Value
0,385.04


## 3. Revenue Trend Analysis

Monthly revenue is analysed to understand how sales performance changes over time.

The analysis groups transactions by year and month and calculates the corresponding revenue.

This helps identify:

- Revenue growth and decline periods
- Seasonal patterns
- High-performing months
- Changes in sales activity over the observation period

In [27]:
query = """
SELECT
    YearMonth,
    ROUND(SUM(Sales), 2) AS Revenue
FROM online_retail
WHERE TransactionType = 'Sale'
GROUP BY YearMonth
ORDER BY YearMonth;
""";

monthly_revenue_sql = pd.read_sql(query, conn)

monthly_revenue_sql

,YearMonth,Revenue
0,2010-12,821452.73
1,2011-01,689811.61
2,2011-02,522545.56
3,2011-03,716215.26
4,2011-04,536968.49
5,2011-05,769296.61
6,2011-06,760547.01
7,2011-07,718076.12
8,2011-08,735717.26
9,2011-09,1056435.19


In [29]:
query = """
SELECT
    YearMonth,
    ROUND(SUM(Sales), 2) AS Revenue
FROM online_retail
WHERE TransactionType = 'Sale'
GROUP BY YearMonth
ORDER BY Revenue DESC
LIMIT 1;
"""

pd.read_sql(query, conn)

,YearMonth,Revenue
0,2011-11,1503866.78


In [31]:
query = """
SELECT
    YearMonth,
    ROUND(SUM(Sales), 2) AS Revenue
FROM online_retail
WHERE TransactionType = 'Sale'
GROUP BY YearMonth
ORDER BY Revenue ASC
LIMIT 1;
"""

pd.read_sql(query, conn)

,YearMonth,Revenue
0,2011-02,522545.56


## 4. Product Performance Analysis

This section evaluates product-level sales performance.

Non-product transaction codes such as postage, bank charges, and other service-related codes are excluded where appropriate so that product performance reflects actual merchandise sales.

The analysis focuses on:

- Top products by revenue
- Top products by quantity
- Product-level order activity
- Revenue contribution of products

This helps identify the products that contribute most to overall sales.

In [33]:
query = """
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Quantity_Sold,
    ROUND(SUM(Sales), 2) AS Revenue
FROM online_retail
WHERE TransactionType = 'Sale'
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10;
"""

top_products_sql = pd.read_sql(query, conn)

top_products_sql

,StockCode,Description,Quantity_Sold,Revenue
0,DOT,DOTCOM POSTAGE,708,206248.77
1,22423,REGENCY CAKESTAND 3 TIER,13862,174156.54
2,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37584,104284.24
4,47566,PARTY BUNTING,18287,99445.23
5,85099B,JUMBO BAG RED RETROSPOT,48375,94159.81
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
7,POST,POSTAGE,3150,78101.88
8,M,Manual,6990,77750.27
9,23084,RABBIT NIGHT LIGHT,30739,66870.03


In [35]:
non_product_codes = [
    "DOT",
    "POST",
    "M",
    "AMAZONFEE",
    "CRUK",
    "BANK CHARGES"
]

In [37]:
query = """
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Quantity_Sold,
    ROUND(SUM(Sales), 2) AS Revenue
FROM online_retail
WHERE TransactionType = 'Sale'
  AND StockCode NOT IN (
      'DOT',
      'POST',
      'M',
      'AMAZONFEE',
      'CRUK',
      'BANK CHARGES'
  )
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10;
"""

top_products_sql = pd.read_sql(query, conn)

top_products_sql

,StockCode,Description,Quantity_Sold,Revenue
0,22423,REGENCY CAKESTAND 3 TIER,13862,174156.54
1,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37584,104284.24
3,47566,PARTY BUNTING,18287,99445.23
4,85099B,JUMBO BAG RED RETROSPOT,48375,94159.81
5,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
6,23084,RABBIT NIGHT LIGHT,30739,66870.03
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,19329,64875.59
8,84879,ASSORTED COLOUR BIRD ORNAMENT,36362,58927.62
9,79321,CHILLI LIGHTS,10305,54096.36


In [39]:
query = """
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Quantity_Sold,
    ROUND(SUM(Sales), 2) AS Revenue
FROM online_retail
WHERE TransactionType = 'Sale'
  AND IsAnomaly = 0
  AND StockCode NOT IN (
      'DOT',
      'POST',
      'M',
      'AMAZONFEE',
      'CRUK',
      'BANK CHARGES'
  )
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10;
"""

top_products_adjusted = pd.read_sql(query, conn)

top_products_adjusted

,StockCode,Description,Quantity_Sold,Revenue
0,22423,REGENCY CAKESTAND 3 TIER,13862,174156.54
1,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37584,104284.24
3,47566,PARTY BUNTING,18287,99445.23
4,85099B,JUMBO BAG RED RETROSPOT,48375,94159.81
5,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
6,23084,RABBIT NIGHT LIGHT,30739,66870.03
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,19329,64875.59
8,84879,ASSORTED COLOUR BIRD ORNAMENT,36362,58927.62
9,79321,CHILLI LIGHTS,10305,54096.36


In [41]:
query = """
SELECT
    InvoiceNo,
    StockCode,
    Description,
    Quantity,
    Sales,
    InvoiceDate,
    CustomerID,
    TransactionType,
    IsAnomaly
FROM online_retail
WHERE StockCode = '23843'
ORDER BY InvoiceDate;
"""

pd.read_sql(query, conn)


,InvoiceNo,StockCode,Description,Quantity,Sales,InvoiceDate,CustomerID,TransactionType,IsAnomaly
0,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6,2011-12-09 09:15:00,16446.0,Sale,0
1,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,-168469.6,2011-12-09 09:27:00,16446.0,Return - Anomaly,1


In [43]:
query = """
SELECT
    InvoiceNo,
    StockCode,
    Description,
    Quantity,
    Sales,
    InvoiceDate,
    CustomerID,
    TransactionType,
    IsAnomaly
FROM online_retail
WHERE StockCode = '23166'
ORDER BY InvoiceDate;
"""

pd.read_sql(query, conn)

,InvoiceNo,StockCode,Description,Quantity,Sales,InvoiceDate,CustomerID,TransactionType,IsAnomaly
0,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,77183.60,2011-01-18 10:01:00,12346.0,Sale,0
1,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,-77183.60,2011-01-18 10:17:00,12346.0,Return - Anomaly,1
2,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,99.84,2011-05-12 10:10:00,14646.0,Sale,0
3,552953,23166,MEDIUM CERAMIC TOP STORAGE JAR,4,5.00,2011-05-12 12:11:00,16745.0,Sale,0
4,553005,23166,MEDIUM CERAMIC TOP STORAGE JAR,5,6.25,2011-05-12 16:29:00,14651.0,Sale,0
...,...,...,...,...,...,...,...,...,...
255,581108,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,2.50,2011-12-07 12:16:00,15984.0,Sale,0
256,581219,23166,MEDIUM CERAMIC TOP STORAGE JAR,1,2.46,2011-12-08 09:28:00,NaN,Sale,0
257,581439,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,4.92,2011-12-08 16:30:00,NaN,Sale,0
258,581476,23166,MEDIUM CERAMIC TOP STORAGE JAR,48,49.92,2011-12-09 08:48:00,12433.0,Sale,0


In [45]:
query = """
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Net_Quantity,
    ROUND(SUM(Sales), 2) AS Net_Revenue
FROM online_retail
WHERE StockCode NOT IN (
    'DOT',
    'POST',
    'M',
    'AMAZONFEE',
    'CRUK',
    'BANK CHARGES'
)
AND NOT (
    (InvoiceNo = '581483' AND StockCode = '23843')
    OR
    (InvoiceNo = 'C581484' AND StockCode = '23843')
    OR
    (InvoiceNo = '541431' AND StockCode = '23166')
    OR
    (InvoiceNo = 'C541433' AND StockCode = '23166')
)
GROUP BY StockCode, Description
HAVING Net_Revenue > 0
ORDER BY Net_Revenue DESC
LIMIT 10;
"""

top_products_final = pd.read_sql(query, conn)

top_products_final


,StockCode,Description,Net_Quantity,Net_Revenue
0,22423,REGENCY CAKESTAND 3 TIER,13007,164459.49
1,47566,PARTY BUNTING,18010,98243.88
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,35006,97659.94
3,85099B,JUMBO BAG RED RETROSPOT,47260,92175.79
4,23084,RABBIT NIGHT LIGHT,30631,66661.63
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,18876,63715.24
6,84879,ASSORTED COLOUR BIRD ORNAMENT,36282,58792.42
7,79321,CHILLI LIGHTS,10225,53746.66
8,23298,SPOTTY BUNTING,8381,42030.67
9,22386,JUMBO BAG PINK POLKADOT,20992,41584.43


## 5. Geographic Sales Analysis

This section analyses sales performance across different countries to understand the geographic distribution of revenue and orders.

The analysis calculates revenue, orders, and quantity sold for each country.

The results help identify the major geographic markets and understand how sales are distributed across the countries represented in the dataset.

In [47]:
query = """
SELECT
    Country,
    ROUND(SUM(Sales), 2) AS Revenue,
    COUNT(DISTINCT InvoiceNo) AS Orders,
    COUNT(DISTINCT CustomerID) AS Customers,
    SUM(Quantity) AS Quantity,
    ROUND(
        SUM(Sales) / COUNT(DISTINCT InvoiceNo),
        2
    ) AS Average_Order_Value
FROM online_retail
WHERE TransactionType = 'Sale'
  AND CustomerID IS NOT NULL
GROUP BY Country
ORDER BY Revenue DESC;
"""

country_analysis = pd.read_sql(query, conn)

country_analysis


,Country,Revenue,Orders,Customers,Quantity,Average_Order_Value
0,United Kingdom,7285024.64,16649,3921,4254037,437.57
1,Netherlands,285446.34,95,9,200937,3004.70
2,EIRE,265262.46,260,3,140383,1020.24
3,Germany,228678.40,457,94,119156,500.39
4,France,208934.31,389,87,111429,537.11
5,Australia,138453.81,57,9,84199,2429.01
6,Spain,61558.56,90,30,27944,683.98
7,Switzerland,56443.95,51,21,30083,1106.74
8,Belgium,41196.34,98,25,23237,420.37
9,Sweden,38367.83,36,8,36078,1065.77


In [49]:
country_analysis["Revenue_Percentage"] = (
    country_analysis["Revenue"]
    / country_analysis["Revenue"].sum()
    * 100
).round(2)

country_analysis


,Country,Revenue,Orders,Customers,Quantity,Average_Order_Value,Revenue_Percentage
0,United Kingdom,7285024.64,16649,3921,4254037,437.57,81.97
1,Netherlands,285446.34,95,9,200937,3004.70,3.21
2,EIRE,265262.46,260,3,140383,1020.24,2.98
3,Germany,228678.40,457,94,119156,500.39,2.57
4,France,208934.31,389,87,111429,537.11,2.35
5,Australia,138453.81,57,9,84199,2429.01,1.56
6,Spain,61558.56,90,30,27944,683.98,0.69
7,Switzerland,56443.95,51,21,30083,1106.74,0.64
8,Belgium,41196.34,98,25,23237,420.37,0.46
9,Sweden,38367.83,36,8,36078,1065.77,0.43


In [51]:
country_analysis.head(10)

,Country,Revenue,Orders,Customers,Quantity,Average_Order_Value,Revenue_Percentage
0,United Kingdom,7285024.64,16649,3921,4254037,437.57,81.97
1,Netherlands,285446.34,95,9,200937,3004.70,3.21
2,EIRE,265262.46,260,3,140383,1020.24,2.98
3,Germany,228678.40,457,94,119156,500.39,2.57
4,France,208934.31,389,87,111429,537.11,2.35
5,Australia,138453.81,57,9,84199,2429.01,1.56
6,Spain,61558.56,90,30,27944,683.98,0.69
7,Switzerland,56443.95,51,21,30083,1106.74,0.64
8,Belgium,41196.34,98,25,23237,420.37,0.46
9,Sweden,38367.83,36,8,36078,1065.77,0.43


## 6. CustomerID Coverage Analysis

CustomerID availability is analysed to understand how much of the transaction data can be attributed to identifiable customers.

This is important because customer-level analysis and RFM segmentation require customer identifiers.

The section also highlights the proportion of revenue associated with transactions where CustomerID is unavailable.

In [53]:
query = """
SELECT
    ROUND(SUM(Sales), 2) AS Revenue_Without_CustomerID,
    COUNT(*) AS Transactions_Without_CustomerID
FROM online_retail
WHERE TransactionType = 'Sale'
  AND CustomerID IS NULL;
"""

pd.read_sql(query, conn)

,Revenue_Without_CustomerID,Transactions_Without_CustomerID
0,1732777.79,133322


In [55]:
query = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE WHEN CustomerID IS NULL THEN Sales ELSE 0 END
        ) / SUM(Sales),
        2
    ) AS Revenue_Percentage_Without_CustomerID
FROM online_retail
WHERE TransactionType = 'Sale';
"""

pd.read_sql(query, conn)


,Revenue_Percentage_Without_CustomerID
0,16.32


## 7. Customer Analysis

This section analyses purchasing behaviour at the customer level.

Revenue, order frequency, quantity purchased, and average order value are calculated for individual customers.

The results provide the foundation for understanding customer value and subsequent RFM segmentation.

In [57]:
query = """
SELECT
    CustomerID,
    ROUND(SUM(Sales), 2) AS Revenue,
    COUNT(DISTINCT InvoiceNo) AS Orders,
    SUM(Quantity) AS Quantity,
    ROUND(
        SUM(Sales) / COUNT(DISTINCT InvoiceNo),
        2
    ) AS Average_Order_Value
FROM online_retail
WHERE TransactionType = 'Sale'
  AND CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY Revenue DESC;
"""

customer_analysis = pd.read_sql(query, conn)

customer_analysis.head(10)

,CustomerID,Revenue,Orders,Quantity,Average_Order_Value
0,14646.0,280206.02,74,197491,3786.57
1,18102.0,259657.30,60,64124,4327.62
2,17450.0,194390.79,46,69973,4225.89
3,16446.0,168472.50,2,80997,84236.25
4,14911.0,143711.17,201,80490,714.98
5,12415.0,124914.53,21,77670,5948.31
6,14156.0,117210.08,55,57768,2131.09
7,17511.0,91062.38,31,64549,2937.50
8,16029.0,80850.84,63,40108,1283.35
9,12346.0,77183.60,1,74215,77183.60


In [59]:
customer_analysis.shape

(4339, 5)

In [61]:
customer_analysis["CustomerID"].nunique()

4339

In [63]:
customer_analysis.head(10)

,CustomerID,Revenue,Orders,Quantity,Average_Order_Value
0,14646.0,280206.02,74,197491,3786.57
1,18102.0,259657.30,60,64124,4327.62
2,17450.0,194390.79,46,69973,4225.89
3,16446.0,168472.50,2,80997,84236.25
4,14911.0,143711.17,201,80490,714.98
5,12415.0,124914.53,21,77670,5948.31
6,14156.0,117210.08,55,57768,2131.09
7,17511.0,91062.38,31,64549,2937.50
8,16029.0,80850.84,63,40108,1283.35
9,12346.0,77183.60,1,74215,77183.60


## 8. RFM Customer Segmentation

RFM analysis is used to segment customers based on their purchasing behaviour.

The analysis evaluates three dimensions:

- **Recency (R):** How recently a customer made a purchase.
- **Frequency (F):** How frequently a customer placed orders.
- **Monetary (M):** How much revenue a customer generated.

Customers are scored using quintile-based RFM scoring and then assigned to meaningful customer segments.

Only positive-quantity transactions are considered for RFM analysis so that customer value is based on purchases rather than returns.

In [65]:
query = """
WITH customer_rfm AS (
    SELECT
        CustomerID,

        CAST(
            julianday('2011-12-10') -
            julianday(MAX(InvoiceDate))
            AS INTEGER
        ) AS Recency,

        COUNT(DISTINCT InvoiceNo) AS Frequency,

        ROUND(SUM(Sales), 2) AS Monetary

    FROM online_retail

    WHERE TransactionType = 'Sale'
      AND CustomerID IS NOT NULL

    GROUP BY CustomerID
)

SELECT *
FROM customer_rfm
ORDER BY Monetary DESC;
"""

rfm_sql = pd.read_sql(query, conn)

rfm_sql.head(10)

,CustomerID,Recency,Frequency,Monetary
0,14646.0,1,74,280206.02
1,18102.0,0,60,259657.30
2,17450.0,8,46,194390.79
3,16446.0,0,2,168472.50
4,14911.0,1,201,143711.17
5,12415.0,24,21,124914.53
6,14156.0,9,55,117210.08
7,17511.0,2,31,91062.38
8,16029.0,38,63,80850.84
9,12346.0,325,1,77183.60


In [67]:
rfm_sql.describe()

,CustomerID,Recency,Frequency,Monetary
count,4339.000000,4339.000000,4339.000000,4339.000000
mean,15299.936852,92.041484,4.271952,2048.215923
std,1721.889758,100.007757,7.705493,8984.248352
min,12346.000000,0.000000,1.000000,0.000000
25%,13812.500000,17.000000,1.000000,306.455000
50%,15299.000000,50.000000,2.000000,668.560000
75%,16778.500000,141.500000,5.000000,1660.315000
max,18287.000000,373.000000,210.000000,280206.020000


In [69]:
rfm_sql.shape

(4339, 4)

In [73]:
rfm_sql.isnull().sum()

CustomerID    0
Recency       0
Frequency     0
Monetary      0
dtype: int64

In [95]:
rfm_sql["R_Score"] = pd.qcut(
    rfm_sql["Recency"],
    5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
).astype(int)

rfm_sql["F_Score"] = pd.qcut(
    rfm_sql["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm_sql["M_Score"] = pd.qcut(
    rfm_sql["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [97]:
rfm_sql["RFM_Score"] = (
    rfm_sql["R_Score"].astype(str)
    + rfm_sql["F_Score"].astype(str)
    + rfm_sql["M_Score"].astype(str)
)

In [99]:
def segment_customer(row):

    R = row["R_Score"]
    F = row["F_Score"]
    M = row["M_Score"]

    if R >= 4 and F >= 4 and M >= 4:
        return "High Value"

    elif R >= 4 and F >= 3:
        return "Loyal"

    elif R >= 4:
        return "Recent"

    elif R <= 2 and F >= 3:
        return "At Risk"

    else:
        return "Low Engagement"


rfm_sql["Segment"] = rfm_sql.apply(
    segment_customer,
    axis=1
)
rfm_sql["Segment"].value_counts()

Segment
Low Engagement    1934
High Value         911
At Risk            656
Loyal              528
Recent             310
Name: count, dtype: int64

In [105]:
rfm_sql.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,14646.0,1,74,280206.02,5,5,5,555,High Value
1,18102.0,0,60,259657.30,5,5,5,555,High Value
2,17450.0,8,46,194390.79,5,5,5,555,High Value
3,16446.0,0,2,168472.50,5,2,5,525,Recent
4,14911.0,1,201,143711.17,5,5,5,555,High Value
5,12415.0,24,21,124914.53,4,5,5,455,High Value
6,14156.0,9,55,117210.08,5,5,5,555,High Value
7,17511.0,2,31,91062.38,5,5,5,555,High Value
8,16029.0,38,63,80850.84,3,5,5,355,Low Engagement
9,12346.0,325,1,77183.60,1,1,5,115,Low Engagement


In [101]:
segment_summary = (
    rfm_sql
    .groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

segment_summary

,Customers,Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary
Segment,,,,,
High Value,911,5676143.38,11.846323,11.501647,6230.673304
Low Engagement,1934,1701753.48,142.329369,2.005171,879.913899
At Risk,656,766931.75,151.725610,3.407012,1169.103277
Loyal,528,407712.50,15.695076,2.958333,772.182765
Recent,310,334667.78,17.716129,1.235484,1079.573484


In [103]:
segment_summary["Revenue_Percentage"] = (
    segment_summary["Revenue"]
    / segment_summary["Revenue"].sum()
    * 100
).round(2)

segment_summary

,Customers,Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary,Revenue_Percentage
Segment,,,,,,
High Value,911,5676143.38,11.846323,11.501647,6230.673304,63.87
Low Engagement,1934,1701753.48,142.329369,2.005171,879.913899,19.15
At Risk,656,766931.75,151.725610,3.407012,1169.103277,8.63
Loyal,528,407712.50,15.695076,2.958333,772.182765,4.59
Recent,310,334667.78,17.716129,1.235484,1079.573484,3.77


In [107]:
rfm_sql.groupby("Segment")[["R_Score", "F_Score", "M_Score"]].mean().round(2)

,R_Score,F_Score,M_Score
Segment,,,
At Risk,1.70,3.55,3.01
High Value,4.60,4.73,4.65
Low Engagement,2.10,2.11,2.41
Loyal,4.42,3.42,2.80
Recent,4.32,1.59,2.15


In [109]:
rfm_sql.groupby("Segment")["RFM_Score"].value_counts()

Segment  RFM_Score
At Risk  233          80
         243          58
         132          56
         232          56
         244          54
                      ..
Recent   425           4
         525           3
         415           1
         514           1
         515           1
Name: count, Length: 114, dtype: int64

In [111]:
rfm_sql.sort_values(
    "Monetary",
    ascending=False
).head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,14646.0,1,74,280206.02,5,5,5,555,High Value
1,18102.0,0,60,259657.30,5,5,5,555,High Value
2,17450.0,8,46,194390.79,5,5,5,555,High Value
3,16446.0,0,2,168472.50,5,2,5,525,Recent
4,14911.0,1,201,143711.17,5,5,5,555,High Value
5,12415.0,24,21,124914.53,4,5,5,455,High Value
6,14156.0,9,55,117210.08,5,5,5,555,High Value
7,17511.0,2,31,91062.38,5,5,5,555,High Value
8,16029.0,38,63,80850.84,3,5,5,355,Low Engagement
9,12346.0,325,1,77183.60,1,1,5,115,Low Engagement


In [113]:
def segment_customer(row):

    R = row["R_Score"]
    F = row["F_Score"]
    M = row["M_Score"]

    if R >= 4 and F >= 4 and M >= 4:
        return "High Value"

    elif R >= 3 and F >= 4:
        return "Loyal"

    elif R >= 4:
        return "Recent"

    elif R <= 2 and F >= 3:
        return "At Risk"

    else:
        return "Low Engagement"


rfm_sql["Segment"] = rfm_sql.apply(
    segment_customer,
    axis=1
)

In [115]:
segment_summary = (
    rfm_sql
    .groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
)

segment_summary["Revenue_Percentage"] = (
    segment_summary["Revenue"]
    / segment_summary["Revenue"].sum()
    * 100
).round(2)

segment_summary = segment_summary.sort_values(
    "Revenue",
    ascending=False
)

segment_summary

,Customers,Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary,Revenue_Percentage
Segment,,,,,,
High Value,911,5676143.38,11.846323,11.501647,6230.673304,63.87
Low Engagement,1613,930811.11,160.873528,1.260384,577.068264,10.47
Loyal,525,899858.13,35.845714,5.030476,1714.015486,10.13
At Risk,656,766931.75,151.725610,3.407012,1169.103277,8.63
Recent,634,613464.52,16.933754,1.812303,967.609653,6.90


In [117]:
rfm_sql["Segment"].value_counts()

Segment
Low Engagement    1613
High Value         911
At Risk            656
Recent             634
Loyal              525
Name: count, dtype: int64

In [119]:
rfm_sql[
    rfm_sql["CustomerID"] == 16029
]

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
8,16029.0,38,63,80850.84,3,5,5,355,Loyal


In [121]:
high_value_revenue = segment_summary.loc[
    "High Value", "Revenue"
]

total_customer_revenue = segment_summary["Revenue"].sum()

high_value_share = (
    high_value_revenue / total_customer_revenue * 100
)

print(f"High Value revenue share: {high_value_share:.2f}%")

High Value revenue share: 63.87%


In [123]:
top_segments_revenue = segment_summary.loc[
    ["High Value", "Loyal"],
    "Revenue"
].sum()

top_segments_share = (
    top_segments_revenue
    / segment_summary["Revenue"].sum()
    * 100
)

print(f"High Value + Loyal revenue share: {top_segments_share:.2f}%")

High Value + Loyal revenue share: 73.99%


In [125]:
query = """
SELECT
    MIN(InvoiceDate) AS First_Date,
    MAX(InvoiceDate) AS Last_Date
FROM online_retail
WHERE TransactionType = 'Sale'
  AND CustomerID IS NOT NULL;
"""

pd.read_sql(query, conn)

,First_Date,Last_Date
0,2010-12-01 08:26:00,2011-12-09 12:50:00


## 9. Cohort Retention Analysis

Cohort analysis is used to understand customer retention over time.

Customers are grouped into cohorts based on the month of their first purchase. Their subsequent purchasing activity is then tracked across later months.

The resulting retention analysis helps evaluate how effectively customers continue purchasing after their initial acquisition.

In [127]:
query = """
WITH customer_first_purchase AS (
    SELECT
        CustomerID,
        MIN(InvoiceDate) AS FirstPurchaseDate
    FROM online_retail
    WHERE TransactionType = 'Sale'
      AND CustomerID IS NOT NULL
    GROUP BY CustomerID
)

SELECT
    CustomerID,
    FirstPurchaseDate,
    strftime('%Y-%m', FirstPurchaseDate) AS CohortMonth
FROM customer_first_purchase
ORDER BY CustomerID;
"""

customer_cohort = pd.read_sql(query, conn)

customer_cohort.head(10)

,CustomerID,FirstPurchaseDate,CohortMonth
0,12346.0,2011-01-18 10:01:00,2011-01
1,12347.0,2010-12-07 14:57:00,2010-12
2,12348.0,2010-12-16 19:09:00,2010-12
3,12349.0,2011-11-21 09:51:00,2011-11
4,12350.0,2011-02-02 16:01:00,2011-02
5,12352.0,2011-02-16 12:33:00,2011-02
6,12353.0,2011-05-19 17:47:00,2011-05
7,12354.0,2011-04-21 13:11:00,2011-04
8,12355.0,2011-05-09 13:49:00,2011-05
9,12356.0,2011-01-18 09:50:00,2011-01


In [129]:
customer_cohort["CohortMonth"].value_counts().sort_index()

CohortMonth
2010-12    885
2011-01    417
2011-02    380
2011-03    452
2011-04    300
2011-05    284
2011-06    242
2011-07    188
2011-08    169
2011-09    299
2011-10    358
2011-11    324
2011-12     41
Name: count, dtype: int64

In [132]:
query = """
WITH customer_first_purchase AS (
    SELECT
        CustomerID,
        MIN(InvoiceDate) AS FirstPurchaseDate
    FROM online_retail
    WHERE TransactionType = 'Sale'
      AND CustomerID IS NOT NULL
    GROUP BY CustomerID
)

SELECT
    o.CustomerID,
    o.InvoiceNo,
    o.InvoiceDate,
    o.Sales,
    strftime('%Y-%m', f.FirstPurchaseDate) AS CohortMonth,
    strftime('%Y-%m', o.InvoiceDate) AS PurchaseMonth
FROM online_retail o
JOIN customer_first_purchase f
    ON o.CustomerID = f.CustomerID
WHERE o.TransactionType = 'Sale'
  AND o.CustomerID IS NOT NULL;
"""

cohort_data = pd.read_sql(query, conn)

cohort_data.head()

,CustomerID,InvoiceNo,InvoiceDate,Sales,CohortMonth,PurchaseMonth
0,17850.0,536365,2010-12-01 08:26:00,15.30,2010-12,2010-12
1,17850.0,536365,2010-12-01 08:26:00,20.34,2010-12,2010-12
2,17850.0,536365,2010-12-01 08:26:00,22.00,2010-12,2010-12
3,17850.0,536365,2010-12-01 08:26:00,20.34,2010-12,2010-12
4,17850.0,536365,2010-12-01 08:26:00,20.34,2010-12,2010-12


In [134]:
cohort_data["CohortMonth"] = pd.to_datetime(
    cohort_data["CohortMonth"]
)

cohort_data["PurchaseMonth"] = pd.to_datetime(
    cohort_data["PurchaseMonth"]
)

In [138]:
cohort_data["CohortIndex"] = (
    (cohort_data["PurchaseMonth"].dt.year -
     cohort_data["CohortMonth"].dt.year) * 12
    +
    (cohort_data["PurchaseMonth"].dt.month -
     cohort_data["CohortMonth"].dt.month)
)

In [140]:
cohort_data[
    [
        "CustomerID",
        "CohortMonth",
        "PurchaseMonth",
        "CohortIndex"
    ]
].head(20)

,CustomerID,CohortMonth,PurchaseMonth,CohortIndex
0,17850.0,2010-12-01,2010-12-01,0
1,17850.0,2010-12-01,2010-12-01,0
2,17850.0,2010-12-01,2010-12-01,0
3,17850.0,2010-12-01,2010-12-01,0
4,17850.0,2010-12-01,2010-12-01,0
5,17850.0,2010-12-01,2010-12-01,0
6,17850.0,2010-12-01,2010-12-01,0
7,17850.0,2010-12-01,2010-12-01,0
8,17850.0,2010-12-01,2010-12-01,0
9,13047.0,2010-12-01,2010-12-01,0


In [142]:
cohort_counts = (
    cohort_data
    .groupby(
        ["CohortMonth", "CohortIndex"]
    )["CustomerID"]
    .nunique()
    .reset_index()
)

cohort_counts.head(20)


,CohortMonth,CohortIndex,CustomerID
0,2010-12-01,0,885
1,2010-12-01,1,324
2,2010-12-01,2,286
3,2010-12-01,3,340
4,2010-12-01,4,321
5,2010-12-01,5,352
6,2010-12-01,6,321
7,2010-12-01,7,309
8,2010-12-01,8,313
9,2010-12-01,9,350


In [144]:
cohort_table = cohort_counts.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="CustomerID"
)

cohort_table

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12
CohortMonth,,,,,,,,,,,,,
2010-12-01,885.0,324.0,286.0,340.0,321.0,352.0,321.0,309.0,313.0,350.0,331.0,445.0,235.0
2011-01-01,417.0,92.0,111.0,96.0,134.0,120.0,103.0,101.0,125.0,136.0,152.0,49.0,NaN
2011-02-01,380.0,71.0,71.0,108.0,103.0,94.0,96.0,106.0,94.0,116.0,26.0,NaN,NaN
2011-03-01,452.0,68.0,114.0,90.0,101.0,76.0,121.0,104.0,126.0,39.0,NaN,NaN,NaN
2011-04-01,300.0,64.0,61.0,63.0,59.0,68.0,65.0,78.0,22.0,NaN,NaN,NaN,NaN
2011-05-01,284.0,54.0,49.0,49.0,59.0,66.0,75.0,27.0,NaN,NaN,NaN,NaN,NaN
2011-06-01,242.0,42.0,38.0,64.0,56.0,81.0,23.0,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01,188.0,34.0,39.0,42.0,51.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01,169.0,35.0,42.0,41.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [146]:
cohort_retention = cohort_table.divide(
    cohort_table[0],
    axis=0
) * 100

In [148]:
cohort_retention = cohort_retention.round(2)

cohort_retention

CohortIndex,0,1,2,3,4,5,6,7,8,9,10,11,12
CohortMonth,,,,,,,,,,,,,
2010-12-01,100.0,36.61,32.32,38.42,36.27,39.77,36.27,34.92,35.37,39.55,37.40,50.28,26.55
2011-01-01,100.0,22.06,26.62,23.02,32.13,28.78,24.70,24.22,29.98,32.61,36.45,11.75,NaN
2011-02-01,100.0,18.68,18.68,28.42,27.11,24.74,25.26,27.89,24.74,30.53,6.84,NaN,NaN
2011-03-01,100.0,15.04,25.22,19.91,22.35,16.81,26.77,23.01,27.88,8.63,NaN,NaN,NaN
2011-04-01,100.0,21.33,20.33,21.00,19.67,22.67,21.67,26.00,7.33,NaN,NaN,NaN,NaN
2011-05-01,100.0,19.01,17.25,17.25,20.77,23.24,26.41,9.51,NaN,NaN,NaN,NaN,NaN
2011-06-01,100.0,17.36,15.70,26.45,23.14,33.47,9.50,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01,100.0,18.09,20.74,22.34,27.13,11.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01,100.0,20.71,24.85,24.26,12.43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [150]:
retention_summary = pd.DataFrame({
    "Month_1": cohort_retention[1],
    "Month_3": cohort_retention[3],
    "Month_6": cohort_retention[6],
    "Month_12": cohort_retention[12]
})

retention_summary

,Month_1,Month_3,Month_6,Month_12
CohortMonth,,,,
2010-12-01,36.61,38.42,36.27,26.55
2011-01-01,22.06,23.02,24.70,NaN
2011-02-01,18.68,28.42,25.26,NaN
2011-03-01,15.04,19.91,26.77,NaN
2011-04-01,21.33,21.00,21.67,NaN
2011-05-01,19.01,17.25,26.41,NaN
2011-06-01,17.36,26.45,9.50,NaN
2011-07-01,18.09,22.34,NaN,NaN
2011-08-01,20.71,24.26,NaN,NaN


In [152]:
retention_summary.mean().round(2)

Month_1     20.62
Month_3     23.24
Month_6     24.37
Month_12    26.55
dtype: float64

In [168]:
df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country',
 'Sales',
 'Year',
 'Month',
 'MonthName',
 'YearMonth',
 'TransactionType',
 'IsAnomaly']

## 10. Exporting Analysis Datasets

The final SQL analysis results are exported as CSV files for reuse in the broader project.

These datasets are used as inputs for reporting and visualization, particularly the Power BI dashboard.

Exporting the analytical outputs also keeps the SQL analysis separate from the visualization layer.

In [156]:
cohort_retention.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\cohort_retention.csv")

In [158]:
rfm_sql.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\customer_rfm.csv",index=False)

In [160]:
segment_summary.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\rfm_segment_summary.csv")

In [166]:
monthly_revenue_sql.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\monthly_revenue.csv",index=False)

In [170]:
df.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\online_retail_final.csv",index=False)

In [172]:
top_products_final.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\top_products.csv",index=False)

In [174]:
country_analysis.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\country_analysis.csv",index=False)

In [176]:
customer_analysis.to_csv(r"C:\Users\BIT\OneDrive\Desktop\E-commerce Data Analytics\data\processed\customer_analysis.csv",index=False)

## 11. SQL Analysis Summary

The SQL analysis provides a structured view of the e-commerce business across sales, products, geography, customers, RFM segments, and customer retention.

The resulting analytical datasets are used in the Power BI dashboard to communicate key business insights through interactive visualizations.